In [57]:
import pandas as pd
import numpy as np

#to split train and test
from sklearn.model_selection import train_test_split

#for min max
from sklearn.preprocessing import MinMaxScaler

#for random forest
from sklearn.ensemble import RandomForestClassifier

#for plot
import matplotlib.pyplot as plt

#for decision tree accuracy score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

#for decision tree 
from sklearn.tree import DecisionTreeClassifier

#for feature selection
from sklearn.feature_selection import SelectKBest, chi2

#for target scaler
from sklearn.preprocessing import StandardScaler
import seaborn as sns

#for KNN classifier
from sklearn.neighbors import KNeighborsClassifier

#for pipeline
from sklearn.pipeline import Pipeline, FeatureUnion

#for gridsearch 
from sklearn.model_selection import GridSearchCV

#for logistic regression 
from sklearn.linear_model import LogisticRegression


In [58]:
#1. Import the dataset and ensure that it loaded properly.

url = 'Loan_Train.csv'

df = pd.read_csv(url, encoding='latin1')  # or encoding='utf-8' if needed

# Preview to confirm it's working
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [59]:
#2. Prepare the data for modeling by performing the following steps:

#Drop the column “Loan_ID.”
df = df.drop(columns=["Loan_ID"])

#Drop any rows with missing data.
df = df.dropna()

#change Loan Status to binary
df.replace({"Loan_Status":{'N':0,'Y':1}},inplace=True)

# Verify no missing data at all
print(df.isnull().sum())


#Convert the categorical features into dummy variables.
df = pd.get_dummies(df)   

df.head()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


C:\Users\jowens224\AppData\Local\Temp\ipykernel_24876\1345986381.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"Loan_Status":{'N':0,'Y':1}},inplace=True)


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Gender_Female,Gender_Male,Married_No,Married_Yes,...,Dependents_1,Dependents_2,Dependents_3+,Education_Graduate,Education_Not Graduate,Self_Employed_No,Self_Employed_Yes,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban
1,4583,1508.0,128.0,360.0,1.0,0,False,True,False,True,...,True,False,False,True,False,True,False,True,False,False
2,3000,0.0,66.0,360.0,1.0,1,False,True,False,True,...,False,False,False,True,False,False,True,False,False,True
3,2583,2358.0,120.0,360.0,1.0,1,False,True,False,True,...,False,False,False,False,True,True,False,False,False,True
4,6000,0.0,141.0,360.0,1.0,1,False,True,True,False,...,False,False,False,True,False,True,False,False,False,True
5,5417,4196.0,267.0,360.0,1.0,1,False,True,False,True,...,False,True,False,True,False,False,True,False,False,True


In [60]:
#3.Split the data into a training and test set, where the “Loan_Status” column is the target.

X = df.drop('Loan_Status', axis=1)  # Features (all columns except target)
y = df['Loan_Status']   # Target variable

#split the data into train and test sets. 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X.shape, X_train.shape, X_test.shape)

(480, 20) (384, 20) (96, 20)


In [61]:
#print(y_train.isnull().sum(), y_test.isnull().sum())

In [62]:
#4. Create a pipeline with a min-max scaler and a KNN classifier (see section 15.3 in the Machine Learning with Python Cookbook).

#create min-max scaler
scaler = MinMaxScaler()

#create a KNN Classifier
knn = KNeighborsClassifier()

# Create the pipeline
pipeline = Pipeline([("scaler", scaler), ("knn", knn)])


In [63]:
#5. Fit a default KNN classifier to the data with this pipeline. Report the model accuracy on the test set. Note: Fitting a pipeline model works just like fitting a regular model.

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Predict on the test data
y_pred = pipeline.predict(X_test)

# Calculate and print accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.7812


In [64]:
#6. Create a search space for your KNN classifier where your “n_neighbors” parameter varies from 1 to 10. (see section 15.3 in the Machine Learning with Python Cookbook).

param_grid = {
    'knn__n_neighbors': list(range(1, 11))  # 1 through 10
}

In [65]:
#7. Fit a grid search with your pipeline, search space, and 5-fold cross-validation to find the best value for the “n_neighbors” parameter.

# Set up GridSearchCV with 5-fold CV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1 
)

In [66]:
#8. Find the accuracy of the grid search best model on the test set. Note: It is possible that this will not be an improvement over the default model, but likely it will be.

# Fit the grid search on the training data
grid_search.fit(X_train, y_train)

# Best parameter and best score
print(f"Best n_neighbors: {grid_search.best_params_['knn__n_neighbors']}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Predict on the test set using the best estimator
y_pred_best = grid_search.best_estimator_.predict(X_test)

# Calculate test set accuracy
test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"Test set accuracy with best KNN: {test_accuracy:.4f}")

Best n_neighbors: 9
Best cross-validation accuracy: 0.7057
Test set accuracy with best KNN: 0.7500


In [67]:
#9.Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values in section 12.3 of the Machine Learning with Python Cookbook.

pipeline = Pipeline([
    ('scaler', MinMaxScaler()),  # Keep scaler for all models
    ('clf', KNeighborsClassifier())  # Placeholder, will be overridden in param grid
])


In [73]:
#define expanded param for hyperparameter tuning 

param_grid = [
    {'clf': [KNeighborsClassifier()],
        'clf__n_neighbors': list(range(1, 11))}, #number of neighbors to use 1-10
    {'clf': [LogisticRegression(max_iter=1000)], #increases the number of iterations
        'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]},
    {'clf': [RandomForestClassifier()],
        'clf__n_estimators': [10, 50, 100], #number of trees
        'clf__max_depth': [None, 5, 10]}] #depth of trees

In [74]:
#create and fit grid search using cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1)

# Fit on training data - fits all combinations of the models and parameters on the training data- cross-validation is used for each 
grid_search.fit(X_train, y_train)


,estimator,Pipeline(step...lassifier())])
,param_grid,"[{'clf': [KNeighborsClassifier()], 'clf__n_neighbors': [1, 2, ...]}, {'clf': [LogisticRegre...max_iter=1000)], 'clf__C': [0.01, 0.1, ...]}, ...]"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,feature_range,"(0, ...)"


In [75]:
#10. What are the best model and hyperparameters found in the grid search? Find the accuracy of this model on the test set.
print(f"Best model: {type(grid_search.best_estimator_.named_steps['clf']).__name__}")
print(f"Best hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

Best model: LogisticRegression
Best hyperparameters: {'clf': LogisticRegression(max_iter=1000), 'clf__C': 10.0}
Best cross-validation accuracy: 0.8100


In [76]:
#Find the accuracy of this model on the test set.

# Predict on the test set using the best model
y_pred = grid_search.predict(X_test)

# Compute test set accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test set accuracy: {test_accuracy:.4f}")


Test set accuracy: 0.8229


#11. Summarize your results.

During this excercise, I tested three models to predict whether a loan will be approved or not based on the provided features. K-Nearest neighbor, Logistic Regression and Random forest. The most accuate model was Logistic regression with a cross-validation accuracy of .81 and a test data accuracy of .82